## Example Usage of History Including Load and Save

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."); io = devnull)
VERSION < v"1.11" && Pkg.develop(path = joinpath(@__DIR__, "..", ".."); io = devnull)   # [sources] needs Julia ≥ 1.11
Pkg.instantiate(; io = devnull)
using FITToolbox;

In [2]:
# Create a Domain
MyDomain = create_domain([100,100,100],[1,1,1];units="cm",σ=0,ε_r=1,μ_r=1); 

# Get the centre of the domain to orient the objects
x_center = MyDomain.nodes_u[end] / 2 * 100 # convert m to cm
y_center = MyDomain.nodes_v[end] / 2 * 100 # convert m to cm
z_center = MyDomain.nodes_w[end] / 2 * 100 # convert m to cm

# Just Randum radii and coil currents
coil_current = 1 #A
coil_radius = 10.0 #cm
# Generate a circular coil with a REAL current at centre
create_circular_loop_source!(MyDomain,coil_radius,x_center,y_center,z_center,DirX();units="cm", current=coil_current, name="source");
# Generate a circular coild with a COMPLEX current at centre - 10 cm offset
create_circular_loop_source!(MyDomain,coil_radius,x_center-10,y_center,z_center,DirX();units="cm", current=complex(coil_current, coil_current), name="source_complex");

# Generate a Brick made of copper
create_brick!(MyDomain, x_center + 10 , 10, 10, 10, 80, 80; units="cm", σ=5.8e7, ε_r=1.0, μ_r=1.0, name="shield", color="copper")
# Generate a Sphere made of copper
create_sphere!(MyDomain, x_center - 25, y_center, z_center, 20; units="cm", σ=5.8e7, ε_r=1.0, μ_r=1.0);

# List all objects registered at the domain
list_objects(MyDomain)

4 objects:
  obj_id  type          name            geometry (m)                                    material
  1       CircularLoop  source          center=(0.5, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 A
  2       CircularLoop  source_complex  center=(0.4, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 + 1.0im A
  3       Brick         shield          origin=(0.6, 0.1, 0.1) lengths=(0.1, 0.8, 0.8)  σ=5.8e7 ε_r=1.0 μ_r=1.0 color=copper
  4       Sphere        obj4            center=(0.25, 0.5, 0.5) radius=0.2              σ=5.8e7 ε_r=1.0 μ_r=1.0

In [3]:
# The Sphere did not get a propper name
# Undo last action
undo!(MyDomain)
list_objects(MyDomain)

3 objects:
  obj_id  type          name            geometry (m)                                    material
  1       CircularLoop  source          center=(0.5, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 A
  2       CircularLoop  source_complex  center=(0.4, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 + 1.0im A
  3       Brick         shield          origin=(0.6, 0.1, 0.1) lengths=(0.1, 0.8, 0.8)  σ=5.8e7 ε_r=1.0 μ_r=1.0 color=copper

In [4]:
# Regenerate Sphere with propper name
create_sphere!(MyDomain, x_center+25, y_center, z_center, 20; name="sphere", units="cm", σ=5.8e7, ε_r=1.0, μ_r=1.0);
list_objects(MyDomain)

4 objects:
  obj_id  type          name            geometry (m)                                    material
  1       CircularLoop  source          center=(0.5, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 A
  2       CircularLoop  source_complex  center=(0.4, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 + 1.0im A
  3       Brick         shield          origin=(0.6, 0.1, 0.1) lengths=(0.1, 0.8, 0.8)  σ=5.8e7 ε_r=1.0 μ_r=1.0 color=copper
  5       Sphere        sphere          center=(0.75, 0.5, 0.5) radius=0.2              σ=5.8e7 ε_r=1.0 μ_r=1.0

In [5]:
# Save the Domain entries to a toml file
save_domain("MyDomain_Example.toml", MyDomain)

# Generate empty Domain
MyDomain = create_domain([100,100,100],[1,1,1];units="cm",σ=0,ε_r=1,μ_r=1); 
list_objects(MyDomain)

0 objects

In [6]:
# Reload previously save Domain
MyDomain = load_domain("MyDomain_Example.toml")
list_objects(MyDomain)

4 objects:
  obj_id  type          name            geometry (m)                                    material
  1       CircularLoop  source          center=(0.5, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 A
  2       CircularLoop  source_complex  center=(0.4, 0.5, 0.5) radius=0.1 normal=DirX   I=1.0 + 1.0im A
  3       Brick         shield          origin=(0.6, 0.1, 0.1) lengths=(0.1, 0.8, 0.8)  σ=5.8e7 ε_r=1.0 μ_r=1.0 color=copper
  5       Sphere        sphere          center=(0.75, 0.5, 0.5) radius=0.2              σ=5.8e7 ε_r=1.0 μ_r=1.0